# 09 · Revoca dei certificati di un device
*Blocco 3 · Certificati. Passo di esecuzione. **Distruttivo, ma solo per questo device**: `device-da-revocare` è dedicato a questa demo.*

Il device si registra e ha un certificato valido. Poi lo revochiamo, dal servizio: il certificato ha ancora giorni di vita, ma l'Hub non lo deve più accettare.

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Fabric: valori dalla Eventstream creata dal 17a (417bd75b-4747-4eff-9d6e-d00ca5964001)

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

## Il device con un certificato valido
Il device parte da una cartella vuota e si registra.

In [ ]:
if (-not $cfg -or -not $out) { throw "Mancano le variabili dello scenario: esegui prima la cella di caricamento (la prima cella di codice) di questo notebook." }
$id = 'device-da-revocare'
$dev = Open-ScenarioDevice -Config $cfg -DeviceId $id -Notebook '09-device-revoca' -Fresh
Invoke-DeviceTick $dev
"Hub assegnato: $($dev.AssignedHub) | certificato $((Get-DeviceCertStatus $dev).Thumbprint)..."

Hub assegnato: iot-iot26-dev-uks-002.azure-devices.net | certificato D3036861B8A5...

## Prima della revoca
L'Hub accetta il certificato.

In [ ]:
Show-HubAcceptance -Label "$id, prima della revoca" -Hub $dev.AssignedHub -DeviceId $id -StateDir $dev.Dir

device-da-revocare, prima della revoca               HTTP 204  (accettato)

## La revoca
`Revoke-AdrDeviceCertificates` invalida i certificati già emessi a questo device. Dura circa mezzo minuto. Richiede un profilo di autenticazione con **certificato firmato da una CA**: quello di ADR.

In [ ]:
$profilo = az iot adr ns device auth list --device-name $id --namespace $cfg.AdrName --resource-group $cfg.Rg --query '[0].name' -o tsv --only-show-errors
Assert-Az "Profilo di autenticazione di $id"
"profilo: $profilo"
Revoke-AdrDeviceCertificates -Name $id -Namespace $cfg.AdrName -ResourceGroup $cfg.Rg -Profile $profilo
"certificati di $id revocati"

profilo: default-121e033fe5de8bf44070ae82daea9dc0

certificati di device-da-revocare revocati

Prossimo passo: **10**, per vedere cosa risponde l'Hub adesso.